# Lineare Algebra – Übungsaufgaben zu NumPy (Teil 1)
Donat Adams

Aufgaben zu den **Kapiteln 1–7**. Die Hashcodes `[#…]` entsprechen den Übungsserien.
Aufgaben mit **NEU** sind zusätzliche Brückenaufgaben, die in den Serien nicht vorkommen.

Jede Aufgabe hat denselben Aufbau:

1. Aufgabentext (wie in der Serie),
2. **Neue Befehle**: nur die Befehle, die in dieser Aufgabe zum ersten Mal vorkommen,
3. aufklappbare Tipps zu jedem neuen Befehl (anklicken),
4. eine Code-Zelle mit den Daten als NumPy-Arrays – Ihre Lösung schreiben Sie unter `# Ihre Lösung`.

### Hilfsfunktionen (Zelle ausführen, nicht ändern)

* `rref(A)` bringt die Matrix `A` in **Zeilenstufenform** und gibt `(R, r)` zurück: die Matrix `R` und den Rang `r`.
  Die Pivots werden nicht auf 1 normiert.
* `null(A)` gibt eine Basis des **Nullraums** von `A` zurück: die *Spalten* der Ergebnismatrix lösen $A\vec x=\vec 0$.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)   # schönere Ausgabe

def rref(A, tol=1e-12):
    """Zeilenstufenform von A (mit Spaltenpivotierung). Gibt (R, Rang) zurück."""
    R = np.array(A, dtype=float)              # Kopie, A bleibt unverändert
    m, n = R.shape
    eps = tol * max(1.0, np.abs(R).max(initial=0.0))
    r = 0
    for c in range(n):
        if r == m:
            break
        p = r + np.argmax(np.abs(R[r:, c]))   # Pivot: betragsgrösstes Element
        if abs(R[p, c]) <= eps:
            R[r:, c] = 0.0                    # numerisches Rauschen entfernen
            continue
        R[[r, p]] = R[[p, r]]
        R[r+1:, c:] -= np.outer(R[r+1:, c] / R[r, c], R[r, c:])
        R[r+1:, c] = 0.0
        r += 1
    return R, r

def null(A, tol=1e-12):
    """Basis des Nullraums {x : A x = 0} als Spalten einer (n x k)-Matrix."""
    R, r = rref(A, tol)
    n = R.shape[1]
    eps = tol * max(1.0, np.abs(R).max(initial=0.0))
    piv = [int(np.argmax(np.abs(R[i]) > eps)) for i in range(r)]   # Pivotspalten
    frei = [c for c in range(n) if c not in piv]
    N = np.zeros((n, len(frei)))
    N[frei, range(len(frei))] = 1.0                      # freie Variablen = Einheitsvektoren
    for i in range(r - 1, -1, -1):                       # Rücksubstitution
        p = piv[i]
        N[p] = -R[i, p+1:] @ N[p+1:] / R[i, p]
    return N


# Einstieg: Vektoren als NumPy-Arrays

## Rechnen mit Vektoren I [#GD49VQ]

$\vec A=\begin{pmatrix}2\\3\end{pmatrix},\;\vec B=\begin{pmatrix}6\\-1\end{pmatrix},\;\vec C=\begin{pmatrix}-4\\3\end{pmatrix},\;\vec D=\begin{pmatrix}-2\\-5\end{pmatrix}$

Berechnen Sie die Komponenten der folgenden Vektoren:
a) $\vec A+\vec B$, b) $\vec C-\vec B$, c) $\vec A-\vec B+\vec C$, d) $\vec A+\vec B-\vec C-\vec D$,
e) $2\cdot\vec A$, f) $(-1)\cdot\vec B$, g) $7\vec A-2\vec B+2\vec C$, h) $\frac72\vec A+\frac13\vec D$

**Neue Befehle:** `np.array`, `+  -  *  /`, `print`

<details><summary><code>np.array</code> – Vektor bzw. Matrix erzeugen</summary>

Ein Vektor ist eine Liste von Zahlen in eckigen Klammern:
```python
a = np.array([1, 2, 3])          # Vektor mit 3 Komponenten
M = np.array([[1, 2], [3, 4]])   # Matrix: eine Liste von Zeilen
a.shape                          # (3,)   M.shape ist (2, 2)
a[0]                             # erste Komponente (Zählung ab 0)
```

</details>

<details><summary><code>+  -  *  /</code> – Rechnen mit Arrays</summary>

Die Rechenzeichen wirken **komponentenweise**; eine Zahl wirkt auf jede Komponente:
```python
a + b        # Vektoraddition
2 * a        # Skalarmultiplikation
a / 3        # jede Komponente durch 3
```

</details>

<details><summary><code>print</code> – Ergebnis ausgeben</summary>

`print('a)', A + B)` gibt Text und Ergebnis nebeneinander aus. Ohne `print` zeigt Jupyter nur den Wert der *letzten* Zeile.

</details>


In [ ]:
A = np.array([2, 3])
B = np.array([6, -1])
C = np.array([-4, 3])
D = np.array([-2, -5])

# Ihre Lösung


# 1 Eliminationsverfahren I

## Gauss-Eliminationsverfahren von Hand [#9T84AZ]

$\vec a=\begin{pmatrix}1\\3\\2\end{pmatrix},\;\vec b=\begin{pmatrix}2\\4\\7\end{pmatrix},\;\vec c=\begin{pmatrix}3\\15\\-3\end{pmatrix}$

Bestimmen Sie, ob die Vektoren $\vec a,\,\vec b$ und $\vec c$ linear abhängig sind.
Falls ja: Geben Sie eine Linearkombination an, welche den Nullvektor ergibt.

*Vorgehen:* Schreiben Sie die Vektoren als Zeilen in eine Matrix und eliminieren Sie mit Zeilenoperationen
(zuerst die $x$-Komponente mit Hilfe von $\vec a$, dann die $y$-Komponente mit Hilfe der zweiten Zeile).

**Neue Befehle:** `np.vstack`, `A[i]`, `A[i] = ...`, `.astype(float)`

<details><summary><code>np.vstack</code> – Arrays zeilenweise stapeln</summary>

```python
np.vstack([a, b, c])     # Matrix mit den Zeilen a, b, c
```

</details>

<details><summary><code>A[i]</code> – Zeile i einer Matrix</summary>

Die Zählung beginnt bei 0: `A[0]` ist die erste, `A[1]` die zweite Zeile.

</details>

<details><summary><code>A[i] = ...</code> – Zeilenoperation</summary>

Die Zeile wird mit dem neuen Wert überschrieben:
```python
A[1] = A[1] - 2*A[0]     # Zeile 1 minus 2 mal Zeile 0
```
Eine Zeilenoperation verändert `A` direkt. Eine Rechnung mit Brüchen braucht Kommazahlen, siehe `.astype(float)`.

</details>

<details><summary><code>.astype(float)</code> – ganze Zahlen in Kommazahlen umwandeln</summary>

`np.vstack([a, b, c]).astype(float)` erzeugt eine Matrix aus Kommazahlen. Sonst bleiben alle Einträge ganzzahlig und Brüche würden abgeschnitten.

</details>


In [ ]:
a = np.array([1, 3, 2])
b = np.array([2, 4, 7])
c = np.array([3, 15, -3])

# Ihre Lösung
A = np.vstack([a, b, c]).astype(float)   # Zeilen: a, b, c
# Zeile 1 (b) minus 2 mal Zeile 0 (a):
# A[1] = ...


## Elimination automatisch: rref und null [#0SAXA8] · NEU (nicht in den Serien)

Dieselben Vektoren $\vec a,\,\vec b,\,\vec c$ wie oben. Jetzt übernimmt die Funktion `rref` die Elimination.

a) Wie gross ist der Rang? Wie viele Nullzeilen bleiben übrig?
b) Bestimmen Sie mit `null` die Koeffizienten $x_1,x_2,x_3$ mit $x_1\vec a+x_2\vec b+x_3\vec c=\vec 0$.

**Neue Befehle:** `rref(A)`, `.T`, `null(A)`, `A @ x`

<details><summary><code>rref(A)</code> – Zeilenstufenform und Rang</summary>

Macht automatisch, was Sie in der letzten Aufgabe von Hand getan haben (inklusive Zeilentausch):
```python
R, r = rref(A)     # R: Zeilenstufenform, r: Rang (Anzahl Pivots)
```
Rang $<$ Anzahl Vektoren heisst: linear abhängig.

</details>

<details><summary><code>.T</code> – Transponierte (Zeilen und Spalten tauschen)</summary>

`V.T` hat die Zeilen von `V` als Spalten.

</details>

<details><summary><code>null(A)</code> – Lösungen von A x = 0</summary>

Die *Spalten* von `N = null(A)` sind Lösungen von $A\vec x=\vec 0$. Weil $A\vec x$ eine Linearkombination der **Spalten** von $A$ ist, setzen Sie die Vektoren als Spalten ein: `null(V.T)` für `V = np.vstack([a, b, c])`.

</details>

<details><summary><code>A @ x</code> – Matrixprodukt</summary>

`A @ x` bildet für jede Zeile von `A` das Skalarprodukt mit `x`. Das Ergebnis ist die Linearkombination der Spalten von `A` mit den Koeffizienten aus `x`.

</details>


In [ ]:
a = np.array([1, 3, 2])
b = np.array([2, 4, 7])
c = np.array([3, 15, -3])

# Ihre Lösung
V = np.vstack([a, b, c])
# R, r = rref(V)


## Gauss-Eliminationsverfahren [#854654]

Bestimmen Sie, ob die Vektoren $\vec a,\,\vec b$ und $\vec c$ linear abhängig sind.
Falls ja: Geben Sie eine Linearkombination an, welche den Nullvektor ergibt.
Die sechs Fälle a)–f) stehen im Dictionary `aufgaben`.

**Neue Befehle:** `for name, (a, b, c) in aufgaben.items():`, `np.linalg.matrix_rank`, `print(f'...')`

<details><summary><code>for name, (a, b, c) in aufgaben.items():</code> – Schleife über alle Fälle</summary>

Ein `dict` hat Schlüssel (`'a'`, `'b'`, …) und Werte (hier: je drei Vektoren). Die Schleife nimmt einen Fall nach dem anderen und entpackt die drei Vektoren:
```python
for name, (a, b, c) in aufgaben.items():
    ...                   # eingerückt: wird für jeden Fall ausgeführt
```

</details>

<details><summary><code>np.linalg.matrix_rank</code> – Rang einer Matrix</summary>

`np.linalg.matrix_rank(V)` liefert den Rang, also dasselbe `r` wie `rref(V)`.

</details>

<details><summary><code>print(f&#x27;...&#x27;)</code> – Text mit Werten</summary>

```python
print(f'{name}) Rang {r}')    # in {...} stehen Variablen
```

</details>


In [ ]:
aufgaben = {
    'a': (np.array([1, 6, 1]),    np.array([2, 14, 1]),  np.array([1, 2, 3])),
    'b': (np.array([1, 5, 2]),    np.array([3, 16, 1]),  np.array([-1, -7, 9])),
    'c': (np.array([8, -7, 6]),   np.array([0, 3, -8]),  np.array([-32, 34, -41])),
    'd': (np.array([4, -3, 0]),   np.array([0, 9, -5]),  np.array([24, -100, 44])),
    'e': (np.array([-4, -2, -9]), np.array([8, 8, -1]),  np.array([38, 29, 38])),
    'f': (np.array([-5, 0, -8]),  np.array([0, -4, -3]), np.array([-25, 12, -31])),
}

# Ihre Lösung
for name, (a, b, c) in aufgaben.items():
    V = np.vstack([a, b, c])
    # Rang bestimmen; falls < 3: Koeffizienten mit null(V.T)


## Einsetzen in die Dreiecksform [#YTJXEP]

Lösen Sie das Gleichungssystem durch Einsetzen von unten nach oben.
$$\left|\begin{array}{rrrcr}
x&+3y&+2z&=&17\\
&3y&+6z&=&24\\
&&4z&=&12
\end{array}\right|$$

**Neue Befehle:** `np.zeros(n)`, `range(n-1, -1, -1)`, `A[i, i+1:]`, `np.allclose(u, v)`

<details><summary><code>np.zeros(n)</code> – Array voller Nullen</summary>

`x = np.zeros(3)` reserviert Platz für die Lösung; die Einträge werden nacheinander überschrieben.

</details>

<details><summary><code>range(n-1, -1, -1)</code> – rückwärts zählen</summary>

`range(2, -1, -1)` liefert 2, 1, 0 – so läuft die Schleife von der letzten Zeile zur ersten:
```python
for i in range(n-1, -1, -1):
    ...
```

</details>

<details><summary><code>A[i, i+1:]</code> – Teilstück einer Zeile</summary>

`A[i, i+1:]` sind die Einträge der Zeile `i` **rechts** vom Diagonalelement. Mit dem gleich langen Stück `x[i+1:]` der bereits bekannten Lösungen gibt `A[i, i+1:] @ x[i+1:]` die Summe der schon bekannten Terme.

</details>

<details><summary><code>np.allclose(u, v)</code> – Gleichheit bis auf Rundungsfehler</summary>

Liefert `True`, wenn alle Einträge praktisch gleich sind. Probe: `np.allclose(A @ x, b)`.

</details>


In [ ]:
A = np.array([[1, 3, 2],
              [0, 3, 6],
              [0, 0, 4]], dtype=float)
b = np.array([17, 24, 12], dtype=float)

# Ihre Lösung
n = len(b)
x = np.zeros(n)
for i in range(n-1, -1, -1):
    pass            # x[i] = (b[i] - bekannte Terme) / A[i, i]


## Dreiecksform [#Q85JSG]

Bestimmen Sie für das lineare Gleichungssystem die Dreiecksform mit dem Gaussverfahren.
Lösen Sie dann das Gleichungssystem durch Einsetzen von unten nach oben.
$$\left|\begin{array}{rrrcr}
x&+2y&+z&=&3\\
-2x&-3y&+z&=&2\\
3x&+y&-2z&=&-1
\end{array}\right|$$

**Neue Befehle:** `np.column_stack`, `R[:, :3]  und  R[:, 3]`, `np.linalg.solve(A, b)`

<details><summary><code>np.column_stack</code> – Arrays spaltenweise nebeneinander</summary>

`np.column_stack([A, b])` hängt `b` als letzte Spalte an `A` an: die **erweiterte Koeffizientenmatrix** $(A\,|\,\vec b)$.

</details>

<details><summary><code>R[:, :3]  und  R[:, 3]</code> – Spalten herausschneiden</summary>

Vor dem Komma stehen die Zeilen, nach dem Komma die Spalten; `:` heisst «alle».
`R[:, :3]` sind alle Zeilen, Spalten 0 bis 2 (die Koeffizientenmatrix); `R[:, 3]` ist die letzte Spalte (rechte Seite).

</details>

<details><summary><code>np.linalg.solve(A, b)</code> – A x = b lösen</summary>

Löst das Gleichungssystem direkt (für quadratisches, lösbares $A$) – wie Ihre Schleife in der letzten Aufgabe.

</details>


In [ ]:
A = np.array([[1, 2, 1],
              [-2, -3, 1],
              [3, 1, -2]], dtype=float)
b = np.array([3, 2, -1], dtype=float)

# Ihre Lösung
Ab = np.column_stack([A, b])
# R, r = rref(Ab)


## Lösen von LGS, Reihenfolge der Gleichungen vertauschen [#FZJIRV]

Bestimmen Sie für das lineare Gleichungssystem die Dreiecksform mit dem Gaussverfahren.
Lösen Sie dann das Gleichungssystem durch Einsetzen von unten nach oben.
$$\left|\begin{array}{rrrcr}
L_1:\;&-6x&+8y&+2z&=&40\\
L_2:\;&12x&-16y&+2z&=&-32\\
L_3:\;&24x&-33y&-3z&=&-120
\end{array}\right|$$
Welche Gleichungen hat `rref` vertauscht? Vergleichen Sie `Ab` und `R`.

*Keine neuen Befehle – Wiederholung.*


In [ ]:
A1 = np.array([[-6, 8, 2],
               [12, -16, 2],
               [24, -33, -3]], dtype=float)
b1 = np.array([40, -32, -120], dtype=float)

# Ihre Lösung


## Gauss-Elimination, Vertauschen von Gleichungen [#F1NB3C]

Bestimmen Sie die Dreiecksform mit dem Gaussverfahren – diesmal **von Hand** mit Zeilenoperationen
(ein Zeilentausch vereinfacht die Rechnung). Lösen Sie dann das Gleichungssystem und kontrollieren Sie mit `rref`.
$$\left|\begin{array}{rrrcr}
L_1:\;&-19x&+38y&-3z&=&114\\
L_2:\;&15x&-22y&+7z&=&-82\\
L_3:\;&x&-2y&+2z&=&-6
\end{array}\right|$$

**Neue Befehle:** `A[[i, j]] = A[[j, i]]`

<details><summary><code>A[[i, j]] = A[[j, i]]</code> – Zeilen tauschen</summary>

```python
A[[0, 2]] = A[[2, 0]]     # Zeile 0 und Zeile 2 vertauschen
```

</details>


In [ ]:
A2 = np.array([[-19, 38, -3],
               [15, -22, 7],
               [1, -2, 2]], dtype=float)
b2 = np.array([114, -82, -6], dtype=float)

# Ihre Lösung
Ab = np.column_stack([A2, b2])
# Zeile 0 und 2 vertauschen, dann Zeilenoperationen ...


# 2 Der Vektorraum

## Parallelogramm [#DDBD9S]

$\vec A=\begin{pmatrix}1\\2\end{pmatrix},\;\vec B=\begin{pmatrix}3\\-2\end{pmatrix},\;\vec C=\begin{pmatrix}2.25\\-1.3\end{pmatrix},\;\vec D=\begin{pmatrix}0.25\\2.7\end{pmatrix}$

Sind die Punkte die aufeinanderfolgenden Ecken eines Parallelogramms $ABCD$?
a) Lösen Sie die Aufgabe, indem Sie Verbindungsvektoren der Punkte berechnen.
b) Lösen Sie die Aufgabe, indem Sie Seitenlängen berechnen.

**Neue Befehle:** `np.linalg.norm`

<details><summary><code>np.linalg.norm</code> – Länge eines Vektors</summary>

`np.linalg.norm(v)` ist die Länge $\|\vec v\|=\sqrt{v_1^2+v_2^2+\dots}$.

</details>


In [ ]:
A = np.array([1.0, 2.0])
B = np.array([3.0, -2.0])
C = np.array([2.25, -1.3])
D = np.array([0.25, 2.7])

# Ihre Lösung


## Gleichschenklige Dreiecke [#5CE2XT]

Handelt es sich bei $ABC$ um gleichschenklige Dreiecke? Wenn ja, berechnen Sie den Flächeninhalt des Dreiecks.
Die Dreiecke a)–c) stehen im Dictionary `dreiecke`.

**Neue Befehle:** `np.isclose(x, y)`, `np.linalg.det`

<details><summary><code>np.isclose(x, y)</code> – zwei Zahlen praktisch gleich?</summary>

Wie `np.allclose`, aber für einzelne Zahlen: `np.isclose(3.0, 3.0000000001)` ist `True`.

</details>

<details><summary><code>np.linalg.det</code> – Determinante</summary>

Für zwei Vektoren im $\mathbb R^2$ ist $|\det(\vec u,\vec v)|$ die Fläche des aufgespannten Parallelogramms; das Dreieck hat die halbe Fläche:
```python
abs(np.linalg.det(np.column_stack([u, v]))) / 2
```

</details>


In [ ]:
dreiecke = {
    'a': (np.array([1.0, -5.0]), np.array([0.0, 3.0]), np.array([-8.0, 2.0])),
    'b': (np.array([1.0, 4.0]),  np.array([5.0, 2.0]), np.array([7.0, 11.0])),
    'c': (np.array([6.0, 2.75]), np.array([4.0, 3.0]), np.array([7.0, 1.0])),
}

# Ihre Lösung
for name, (A, B, C) in dreiecke.items():
    a, b, c = np.linalg.norm(B - C), np.linalg.norm(C - A), np.linalg.norm(A - B)
    # gleichschenklig, wenn zwei Seiten gleich lang sind


# 4 Funktionen und Trigonometrie

## Polarkoordinaten [#UCWJYR]

Geben Sie die Vektoren in kartesischen Koordinaten an. Länge $r=\|\vec v\|$ und Winkel $\varphi$:

a) $r=1,\ \varphi=45^\circ$, b) $r=2,\ \varphi=-225^\circ$, c) $r=5,\ \varphi=\frac{5\pi}{36}$,
d) $r=2,\ \varphi=60^\circ$, e) $r=\sqrt2,\ \varphi=-\frac\pi4$, f) $r=7,\ \varphi=35^\circ$

**Neue Befehle:** `np.deg2rad`, `np.cos, np.sin`, `def`

<details><summary><code>np.deg2rad</code> – Grad in Bogenmass</summary>

NumPy rechnet im Bogenmass: `np.deg2rad(45)` ist $\pi/4$.

</details>

<details><summary><code>np.cos, np.sin</code> – Kosinus und Sinus</summary>

Funktionieren auch auf ganzen Arrays. Kartesische Koordinaten: $x=r\cos\varphi,\ y=r\sin\varphi$.

</details>

<details><summary><code>def</code> – eigene Funktion definieren</summary>

```python
def verdopple(v):
    return 2 * v
```
Der Wert nach `return` ist das Ergebnis des Aufrufs `verdopple(np.array([1, 2]))`.

</details>


In [ ]:
# Winkel a, b, d, f in Grad; c und e im Bogenmass
r_a, phi_a = 1.0, 45.0
r_b, phi_b = 2.0, -225.0
r_c, phi_c = 5.0, 5*np.pi/36
r_d, phi_d = 2.0, 60.0
r_e, phi_e = np.sqrt(2.0), -np.pi/4
r_f, phi_f = 7.0, 35.0

# Ihre Lösung
def kartesisch(r, phi):       # phi im Bogenmass
    ...


## Symmetrie numerisch testen [#6GT0K4] · NEU (nicht in den Serien)

Eine Funktion heisst *gerade*, wenn $f(-x)=f(x)$, und *ungerade*, wenn $f(-x)=-f(x)$.
Testen Sie numerisch auf 101 Punkten im Intervall $[-5,5]$:
$f(x)=\sin(x^2)$, $g(x)=x^2\sin(x)$, $h(x)=\sin(x)+\cos(x)$.

**Neue Befehle:** `np.linspace`, `lambda x: ...`

<details><summary><code>np.linspace</code> – gleichmässig verteilte Punkte</summary>

`np.linspace(-5, 5, 101)` erzeugt 101 Punkte von $-5$ bis $5$ (inklusive Endpunkte).

</details>

<details><summary><code>lambda x: ...</code> – kurze Funktion</summary>

```python
f = lambda x: np.sin(x**2)     # x**2 ist x hoch 2
f(np.array([0.0, 1.0]))        # wirkt auf alle Einträge
```
`lambda` ist die Kurzform von `def f(x): return ...`.

</details>


In [ ]:
# Ihre Lösung
x = np.linspace(-5, 5, 101)
# f = lambda x: ...


## Symmetrien [#MZ2D7Y]

Bestimmen Sie die Symmetrien der folgenden Funktionen **exakt** (symbolisch): gerade, ungerade oder keines von beiden?
Untersuchen Sie $f(-x)$ und vergleichen Sie mit $f(x)$ bzw. $-f(x)$.

a) $\sin(x^2)$, b) $\sin(x)^2$, c) $\sin(\cos x)$, d) $\cos(x-6\pi)$, e) $x^2\sin x$,
f) $x\cos x$, g) $(1+x^2)\cos x$, h) $\frac{\sin x}{1-x}$, i) $\sin(-x)$, j) $\cos(-x)$

**Neue Befehle:** `import sympy as sym`, `sym.symbols('x')`, `f.subs(x, -x)`, `sym.simplify`

<details><summary><code>import sympy as sym</code> – Computer-Algebra (exakte Rechnung)</summary>

SymPy rechnet mit Formeln statt mit Zahlen.

</details>

<details><summary><code>sym.symbols(&#x27;x&#x27;)</code> – symbolische Variable</summary>

`x = sym.symbols('x')` macht `x` zu einer Variablen. Funktionen heissen `sym.sin`, `sym.cos`, die Zahl $\pi$ heisst `sym.pi`.

</details>

<details><summary><code>f.subs(x, -x)</code> – einsetzen</summary>

`f.subs(x, -x)` ersetzt in der Formel `f` überall `x` durch `-x`.

</details>

<details><summary><code>sym.simplify</code> – vereinfachen</summary>

`sym.simplify(f.subs(x, -x) - f)` ist `0`, wenn $f(-x)=f(x)$ gilt. Vergleichen Sie so auch mit $-f(x)$.

</details>


In [ ]:
import sympy as sym
x = sym.symbols('x')
funktionen = {
    'a': sym.sin(x**2),        'b': sym.sin(x)**2,
    'c': sym.sin(sym.cos(x)),  'd': sym.cos(x - 6*sym.pi),
    'e': x**2 * sym.sin(x),    'f': x * sym.cos(x),
    'g': (1 + x**2) * sym.cos(x),  'h': sym.sin(x) / (1 - x),
    'i': sym.sin(-x),          'j': sym.cos(-x),
}

# Ihre Lösung
for name, f in funktionen.items():
    fm = f.subs(x, -x)          # f(-x)
    # gerade, wenn sym.simplify(fm - f) == 0 ...


# 5 Skalarprodukt

## Skalarprodukt in einer Orthonormalbasis [#9XPSE2]

Berechnen Sie die Skalarprodukte. Was können Sie anhand der Resultate über den Winkel $\varphi$ zwischen den Vektoren sagen?

a) $(1,2,3)\odot(4,-1,2)$, b) $(3,4)\odot(4,-3)$, c) $(2,-1,3,0,5)\odot(1,4,2,7,-1)$,
d) $(0,2,-3)\odot(6,5,8)$, e) $(10,2)\odot(0,1)$, f) $(7,0,4,6,1)\odot(-3,9,2,2,1)$

**Neue Befehle:** `np.dot(u, v)`, `np.sign`

<details><summary><code>np.dot(u, v)</code> – Skalarprodukt</summary>

`np.dot(u, v)` ist $u_1v_1+u_2v_2+\dots$; kurz auch `u @ v`.

</details>

<details><summary><code>np.sign</code> – Vorzeichen</summary>

`np.sign(s)` ist $+1$, $0$ oder $-1$. Vorzeichen $+$: spitzer Winkel, $0$: rechter Winkel, $-$: stumpfer Winkel.

</details>


In [ ]:
paare = {
    'a': (np.array([1, 2, 3]),        np.array([4, -1, 2])),
    'b': (np.array([3, 4]),           np.array([4, -3])),
    'c': (np.array([2, -1, 3, 0, 5]), np.array([1, 4, 2, 7, -1])),
    'd': (np.array([0, 2, -3]),       np.array([6, 5, 8])),
    'e': (np.array([10, 2]),          np.array([0, 1])),
    'f': (np.array([7, 0, 4, 6, 1]),  np.array([-3, 9, 2, 2, 1])),
}

# Ihre Lösung


## Winkel zwischen Vektoren [#520784]

Berechnen Sie den Winkel zwischen den Vektoren $\vec a$ und $\vec b$ (in Grad).

a) $(1,1,-1)$ und $(1,-1,1)$, b) $(1,1)$ und $(1,-1)$, c) $(-1,1)$ und $(1351,362)$, d) $(1,0,1)$ und $(0,1,1)$

**Neue Befehle:** `np.arccos`, `np.degrees`

<details><summary><code>np.arccos</code> – Umkehrfunktion des Kosinus</summary>

Aus $\cos\varphi=\dfrac{\vec a\odot\vec b}{\|\vec a\|\,\|\vec b\|}$ folgt `phi = np.arccos(...)` im Bogenmass.
Wegen Rundungsfehlern kann der Bruch minimal ausserhalb von $[-1,1]$ liegen; dann hilft `np.clip(c, -1, 1)`.

</details>

<details><summary><code>np.degrees</code> – Bogenmass in Grad</summary>

`np.degrees(np.pi/2)` ist `90.0`.

</details>


In [ ]:
paare = {
    'a': (np.array([1, 1, -1]), np.array([1, -1, 1])),
    'b': (np.array([1, 1]),     np.array([1, -1])),
    'c': (np.array([-1, 1]),    np.array([1351, 362])),
    'd': (np.array([1, 0, 1]),  np.array([0, 1, 1])),
}

# Ihre Lösung


## Projektion und Lot [#JIUJV2]

Zerlegen Sie $\vec b$ in die Projektion auf $\vec a$ und das Lot (den Anteil senkrecht zu $\vec a$):
$\vec b=\begin{pmatrix}6\\0\\8\end{pmatrix}$ und $\vec a=\begin{pmatrix}1\\1\\0\end{pmatrix}$.
Kontrollieren Sie, dass Projektion und Lot zusammen $\vec b$ ergeben und das Lot senkrecht auf $\vec a$ steht.

*Keine neuen Befehle – Wiederholung.*


In [ ]:
b = np.array([6.0, 0.0, 8.0])
a = np.array([1.0, 1.0, 0.0])

# Ihre Lösung


## Lot [#DW2PC2]

Fällen Sie das Lot vom Punkt $C$ auf die Gerade $g$ und berechnen Sie den Abstand des Punktes $C$ zur Geraden $g$:

a) $g:\ \vec x=\begin{pmatrix}1\\2\end{pmatrix}+k\begin{pmatrix}4\\3\end{pmatrix}$ und $C=\begin{pmatrix}9\\-2\end{pmatrix}$,
b) $g:\ \vec x=\begin{pmatrix}2\\5\end{pmatrix}+k\begin{pmatrix}1\\-2\end{pmatrix}$ und $C=\begin{pmatrix}6\\7\end{pmatrix}$.

Schreiben Sie eine Funktion `lot(P, v, C)`, die den Lotfusspunkt und den Abstand zurückgibt.

*Keine neuen Befehle – Wiederholung.*


In [ ]:
# (Aufpunkt P, Richtungsvektor v, Punkt C)
faelle = {
    'a': (np.array([1.0, 2.0]), np.array([4.0, 3.0]),  np.array([9.0, -2.0])),
    'b': (np.array([2.0, 5.0]), np.array([1.0, -2.0]), np.array([6.0, 7.0])),
}

# Ihre Lösung
def lot(P, v, C):
    ...      # Rückgabe: Fusspunkt F und Abstand d


## Spiegelung [#109810]

Spiegeln Sie das Dreieck $A=(0,0),\ B=(3,-1),\ C=(2,6)$ an der Geraden
$g:\ \vec x=\begin{pmatrix}7\\-2\end{pmatrix}+k\begin{pmatrix}2\\2\end{pmatrix}$.
Bestimmen Sie zuerst die Projektionen auf $g$, dann die gespiegelten Punkte.

*Vorgehen:* Normalenvektor $\vec n$ von $g$; für jeden Punkt $X$ ist $\vec k=\dfrac{(X-P)\odot\vec n}{\vec n\odot\vec n}\vec n$;
Projektion $X'=X-\vec k$, Spiegelung $X''=X-2\vec k$.

**Neue Befehle:** `[... for X in (A, B, C)]`

<details><summary><code>[... for X in (A, B, C)]</code> – Liste in einer Zeile</summary>

Wendet einen Ausdruck auf jeden Punkt an:
```python
[2 * X for X in (A, B, C)]       # Liste mit drei Vektoren
```

</details>


In [ ]:
A = np.array([0.0, 0.0])
B = np.array([3.0, -1.0])
C = np.array([2.0, 6.0])
Pp = np.array([7.0, -2.0])      # Aufpunkt von g
vv = np.array([2.0, 2.0])       # Richtungsvektor von g

# Ihre Lösung
nn = np.array([-vv[1], vv[0]])      # Normalenvektor
# def teil(X): ...


# 6 Vektorprodukt

## Vektorprodukt [#65KXV2]

Berechnen Sie das Vektorprodukt von $\vec a=\begin{pmatrix}2\\-1\\3\end{pmatrix}$ und $\vec b=\begin{pmatrix}1\\4\\-2\end{pmatrix}$.
Kontrollieren Sie, dass $\vec a\times\vec b$ senkrecht auf $\vec a$ und $\vec b$ steht.

**Neue Befehle:** `np.cross(a, b)`

<details><summary><code>np.cross(a, b)</code> – Vektorprodukt</summary>

`np.cross(a, b)` ist $\vec a\times\vec b$ für Vektoren im $\mathbb R^3$.

</details>


In [ ]:
a = np.array([2, -1, 3])
b = np.array([1, 4, -2])

# Ihre Lösung


## Fläche und Vektorprodukt [#68EML3]

Berechnen Sie die Fläche des Parallelogramms, das von den Vektoren $\vec a$ und $\vec b$ aufgespannt wird
(Fläche $=\|\vec a\times\vec b\|$).

*Keine neuen Befehle – Wiederholung.*


In [ ]:
paare = {
    'a': (np.array([4, -5, 0]),   np.array([5, 1, 0])),
    'b': (np.array([10, -4, 0]),  np.array([0, 0, 17])),
    'c': (np.array([3, -2, 5]),   np.array([18, 4, 0])),
    'd': (np.array([1, -1, -4]),  np.array([19, 14, -3])),
    'e': (np.array([10, 10, 19]), np.array([11, 3, 5])),
}

# Ihre Lösung


## Spatprodukt [#5AHNL8]

Berechnen Sie das Volumen des Parallelepipeds, das von $\vec u,\,\vec v,\,\vec w$ aufgespannt wird
(Volumen $=|\vec u\odot(\vec v\times\vec w)|$). Geben Sie jeweils an, ob die Vektoren linear abhängig sind,
und kontrollieren Sie mit `np.linalg.matrix_rank`.

*Keine neuen Befehle – Wiederholung.*


In [ ]:
# Zeilen: u, v, w
tripel = {
    'a': np.array([[4, 0, 0],   [4, -3, 0],  [1, 1, 2]]),
    'b': np.array([[0, 7, 2],   [0, 20, 0],  [1, -4, 0]]),
    'c': np.array([[4, 17, -5], [0, 3, 19],  [4, 20, 14]]),
    'd': np.array([[20, 13, 2], [-2, -3, 6], [0, 9, -3]]),
}

# Ihre Lösung


## Abstand Punkt-Gerade [#CJ1IXZ]

Wir betrachten die Gerade $\vec X=\vec A+\lambda\cdot\vec v$ und den Punkt $\vec B$ ($\vec A,\vec B,\vec X\in\mathbb R^3$, $\lambda\in\mathbb R$).
Wie gross ist ihr Abstand? Berechnen Sie dazu zuerst die Fläche, die von $\vec r=\vec B-\vec A$ und $\vec v$ aufgespannt wird.

*Keine neuen Befehle – Wiederholung.*


In [ ]:
# (Aufpunkt A, Richtungsvektor v, Punkt B)
faelle = {
    'a': (np.array([-4.0, 0.0, 5.0]), np.array([8.0, 0.0, 15.0]),  np.array([-4.0, 1.0, 3.0])),
    'b': (np.array([7.0, 7.0, 1.0]),  np.array([-12.0, 35.0, 0.0]), np.array([7.0, 12.0, -3.0])),
    'c': (np.array([4.0, -4.0, 7.0]), np.array([6.0, 2.0, 5.0]),    np.array([0.0, -2.0, 9.0])),
}

# Ihre Lösung


# 7 Ebenen in $\mathbb{R}^3$

## Ebene: Normalen- und Koordinatenform [#H4KD9Q]

Geben Sie den Normalenvektor und die Koordinatenform $n_1x+n_2y+n_3z=d$ der folgenden Ebenen an
(mit gekürzten ganzen Zahlen). Kontrollieren Sie den Normalenvektor mit `null`: er löst $\begin{pmatrix}\vec u^{\,T}\\\vec v^{\,T}\end{pmatrix}\vec n=\vec 0$.

a) $E:\ \vec x=\begin{pmatrix}50\\10\\-32\end{pmatrix}+\mu\begin{pmatrix}-40\\9\\0\end{pmatrix}+\nu\begin{pmatrix}40\\-9\\10\end{pmatrix}$
b) $E$ enthält $A=(2,-35,105)$, $B=(7,0,117)$, $C=(-1,35,129)$
c) $E$ enthält $A=(56,13,135)$, $B=(146,18,79)$, $C=(101,8,107)$

**Neue Befehle:** `np.gcd.reduce`

<details><summary><code>np.gcd.reduce</code> – grösster gemeinsamer Teiler einer Liste</summary>

`np.gcd.reduce(np.array([90, 400, 0]))` ist `10`. Teilt man den Normalenvektor (und $d$) durch diese Zahl, erhält man kleinere ganze Zahlen. Die Funktion braucht ganze Zahlen: `n.astype(int)`.

</details>


In [ ]:
# Aufpunkt und zwei Richtungsvektoren (a) bzw. drei Punkte (b, c)
P0 = np.array([50.0, 10.0, -32.0])
u  = np.array([-40.0, 9.0, 0.0])
v  = np.array([40.0, -9.0, 10.0])

A1, B1, C1 = np.array([2.0, -35.0, 105.0]),  np.array([7.0, 0.0, 117.0]),  np.array([-1.0, 35.0, 129.0])
A2, B2, C2 = np.array([56.0, 13.0, 135.0]),  np.array([146.0, 18.0, 79.0]), np.array([101.0, 8.0, 107.0])

# Ihre Lösung


## Abstand Punkt-Ebene in $\mathbb{R}^3$ [#2MZJ54]

Berechnen Sie den Abstand zwischen $E$ und dem Punkt $P$.
Abstand $=\dfrac{|\vec n\odot(P-A)|}{\|\vec n\|}$ mit Aufpunkt $A$ und Normalenvektor $\vec n$.

a) $E:\ \vec x=(50,10,-32)+\mu(21,0,20)+\nu(42,20,40)$, $P=(10,10,10)$
b) $E:\ \vec x=(50,200,-1)+\mu(-40,9,0)+\nu(40,-9,10)$, $P=(5,0,-1)$
c) $E$ enthält $A=(2,-35,105),\,B=(7,0,117),\,C=(-1,35,129)$, $P=(2,1,0)$

**Neue Befehle:** `abs`

<details><summary><code>abs</code> – Betrag</summary>

`abs(x)` ist $|x|$ (auch `np.abs`).

</details>


In [ ]:
# (Aufpunkt A, Richtungsvektoren u, v, Punkt P)
faelle = {
    'a': (np.array([50, 10, -32]), np.array([21, 0, 20]),  np.array([42, 20, 40]), np.array([10, 10, 10])),
    'b': (np.array([50, 200, -1]), np.array([-40, 9, 0]),  np.array([40, -9, 10]), np.array([5, 0, -1])),
    'c': (np.array([2, -35, 105]), np.array([5, 35, 12]),  np.array([-3, 70, 24]), np.array([2, 1, 0])),
}

# Ihre Lösung


## Ebene und Normalenvektor [#7EQKL7]

Geben Sie einen Punkt der Ebene an. Lesen Sie dann den Normalenvektor und den Abstand zum Ursprung aus
und geben Sie die Parameterform der Ebene an.

a) $6x+4y+3z-12=0$, b) $x-y-z+1=0$, c) $5y=0$, d) $6x-3y-2z=-30$, e) $7x=3$, f) $-4x+2y-z+8=0$

*Tipp:* Die Richtungsvektoren sind der Nullraum von $\vec n^{\,T}$ (eine Zeile!).

**Neue Befehle:** `n.reshape(1, 3)`

<details><summary><code>n.reshape(1, 3)</code> – Vektor als Zeile einer Matrix</summary>

`n.reshape(1, 3)` macht aus dem Vektor `n` eine Matrix mit **einer Zeile** und 3 Spalten. So lässt sich `null(n.reshape(1, 3))` aufrufen; das Ergebnis hat zwei Spalten – zwei Richtungsvektoren der Ebene.

</details>


In [ ]:
# Koordinatenform n1 x + n2 y + n3 z = d  ->  (n, d)
ebenen = {
    'a': (np.array([6.0, 4.0, 3.0]),   12.0),
    'b': (np.array([1.0, -1.0, -1.0]), -1.0),
    'c': (np.array([0.0, 5.0, 0.0]),   0.0),
    'd': (np.array([6.0, -3.0, -2.0]), -30.0),
    'e': (np.array([7.0, 0.0, 0.0]),   3.0),
    'f': (np.array([-4.0, 2.0, -1.0]), -8.0),
}

# Ihre Lösung
for name, (n, d) in ebenen.items():
    # Punkt: d * n / (n @ n);  Abstand zum Ursprung: |d| / ||n||
    ...
